# ruEncod: мини-модель с токенизатором o200k в облаке

Ноутбук обучает последнего кандидата сравнения — **`o200k`** (токенизатор GPT-4o) — на облачной видеокарте
и считает **bpb** (бит на байт) на отложенном тексте. Всё нужное внутри: код проекта (тот же, на котором дома
учились `bpe-64k`, `unigram-64k`, `morph-bpe-64k`), конфиги и проверка данных. Репозиторий не нужен.

**Что произойдёт**
1. Проверка видеокарты: bf16 на A100 / L4 / H100 / RTX 30xx и новее, fp16 на T4 / P100.
2. Установка библиотек.
3. Запись кода проекта в папку `ruencode/`.
4. Скачивание и сборка корпуса (~5 ГБ текста с Hugging Face, ~15–30 мин).
5. Проверка, что данные совпали с домашними байт в байт.
6. Пробный прогон: скорость, память и прогноз времени.
7. Обучение `o200k` (~13 900 шагов) и оценка → файл `lm_bpb_o200k.csv`.

**Как запускать**
- **Colab:** Среда выполнения → Сменить среду выполнения → GPU, затем «Выполнить всё».
  При `USE_DRIVE = True` корпус и сохранения лягут на Google Диск (нужно ~8 ГБ свободного места):
  если Colab отключится, снова «Выполнить всё» — сборка пропустится, обучение продолжится
  с последнего сохранения (они делаются раз в 10 минут).
- **Kaggle:** Settings → Accelerator → GPU, Internet → On. Надёжнее всего **Save Version → Save & Run All**:
  ноутбук отработает в фоне (до 12 ч), результат — во вкладке Output.
- Любой другой Jupyter с видеокартой NVIDIA тоже подойдёт.

В конце скачайте `lm_bpb_o200k.csv` и положите в `ruencode/results/` у себя на компьютере.

## 1. Настройки

In [ ]:
CANDIDATE = "o200k"          # кого обучаем
USE_DRIVE = True             # только Colab: корпус и сохранения на Google Диске — переживут отключение
RUN_SMOKE = True             # пробный прогон на 1 МБ текста: скорость и прогноз времени (~3–5 мин)
ALLOW_DATA_MISMATCH = False  # True — обучать, даже если скачанные данные не совпали с домашними

## 2. Видеокарта и папки

In [ ]:
import os, sys
from pathlib import Path

import torch

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").exists()

assert torch.cuda.is_available(), "Видеокарты нет: включите GPU в настройках среды выполнения и перезапустите."
cap = torch.cuda.get_device_capability()
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
precision = "bf16" if cap[0] >= 8 else "fp16"
print(f"{torch.cuda.get_device_name()}: compute {cap[0]}.{cap[1]}, {vram_gb:.0f} ГБ, torch {torch.__version__} → {precision}")
if precision == "fp16":
    print("Карта без bf16: учимся в fp16 с масштабированием loss. Дома кандидаты учились в bf16 — "
          "на bpb это почти не влияет, точность записывается в результат.")

if IN_COLAB:
    WORK = Path("/content/ruencode")
elif IN_KAGGLE:
    WORK = Path("/kaggle/working/ruencode")
else:
    WORK = Path("ruencode").resolve()
DATA_DIR, RESULTS_DIR = WORK / "data", WORK / "results"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/ruencode_cloud/data")
    RESULTS_DIR = Path("/content/drive/MyDrive/ruencode_cloud/results")
elif IN_KAGGLE:
    DATA_DIR = Path("/tmp/ruencode_data")  # корпус в Output не нужен; результаты — в /kaggle/working

for d in ("src/ruencod/data", "src/ruencod/eval", "src/ruencod/lm", "configs"):
    (WORK / d).mkdir(parents=True, exist_ok=True)
for pkg in ("", "data", "eval", "lm"):
    (WORK / "src/ruencod" / pkg / "__init__.py").touch()
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONPATH"] = str(WORK / "src")
sys.path.insert(0, str(WORK / "src"))

# необязательно: токен Hugging Face (секрет HF_TOKEN) ускоряет скачивание
try:
    if IN_COLAB:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    elif IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

import codecs, signal, subprocess


def run(*args):
    '''python -m <модуль> с выводом в ноутбук. При ошибке останавливает «Выполнить всё»;
    кнопка «Стоп» передаётся процессу как Ctrl+C (обучение успеет сохраниться).'''
    p = subprocess.Popen([sys.executable, "-u", "-m", *args], cwd=WORK,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    try:
        while chunk := p.stdout.read1(4096):
            sys.stdout.write(dec.decode(chunk))
            sys.stdout.flush()
    except KeyboardInterrupt:
        p.send_signal(signal.SIGINT)
        p.wait()
        raise
    if p.wait() != 0:
        raise RuntimeError(f"python -m {' '.join(args)} завершился с кодом {p.returncode} — причина в выводе выше")


%cd {WORK}
print("код:", WORK)
print("данные:", DATA_DIR)
print("результаты:", RESULTS_DIR)

## 3. Библиотеки

Версии — как дома: от версии `datasets` зависит порядок документов при потоковом чтении,
а значит, и то, какие тексты попадут в корпус. torch берётся тот, что уже есть в среде.

In [ ]:
!pip install -q "datasets==5.1.0" "huggingface_hub==2.2.0" "tokenizers==0.23.3" "tiktoken==0.14.0" "zstandard==0.25.0" regex pyarrow pyyaml pandas tqdm

## 4. Код проекта

Ячейки ниже только записывают файлы — это копия `src/ruencod` и `configs` из репозитория.
Главное для эксперимента: `lm/model.py` (модель ~85M параметров без эмбеддингов, 12 слоёв × 768),
`lm/train_lm.py` (обучение, сохранения, bf16/fp16), `lm/eval_bpb.py` (bpb), `lm/run.py` (запуск).

In [ ]:
%%writefile src/ruencod/config.py
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]
DEFAULT_CONFIG = ROOT / "configs" / "experiments.yaml"


def _merge(base: dict, over: dict) -> dict:
    out = dict(base)
    for k, v in over.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _merge(out[k], v)
        else:
            out[k] = v
    return out


def load_config(path: str | Path = DEFAULT_CONFIG) -> dict:
    path = Path(path)
    cfg = yaml.safe_load(path.read_text(encoding="utf-8"))
    base = cfg.pop("base", None)
    if base:
        cfg = _merge(load_config(path.parent / base), cfg)
    return cfg


def cfg_path(cfg: dict, key: str) -> Path:
    p = Path(cfg["paths"][key])
    return p if p.is_absolute() else ROOT / p

In [ ]:
%%writefile src/ruencod/jsonl.py
import io
import json
import os
from pathlib import Path
from typing import Iterator

import zstandard as zstd


class ZstJsonlWriter:
    """Пишет во временный файл; на месте `path` он появляется только после успешного закрытия."""

    def __init__(self, path: Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._tmp = self.path.with_name(self.path.name + ".partial")
        self._f = open(self._tmp, "wb")
        self._z = zstd.ZstdCompressor(level=3).stream_writer(self._f)

    def write(self, rec: dict) -> None:
        self._z.write((json.dumps(rec, ensure_ascii=False) + "\n").encode("utf-8"))

    def close(self) -> None:
        self._z.flush(zstd.FLUSH_FRAME)
        self._z.close()
        if not self._f.closed:
            self._f.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, *exc):
        self.close()
        if exc_type is None:
            os.replace(self._tmp, self.path)
        else:
            self._tmp.unlink(missing_ok=True)


def iter_jsonl_zst(path: Path) -> Iterator[dict]:
    with open(path, "rb") as f:
        reader = zstd.ZstdDecompressor().stream_reader(f)
        for line in io.TextIOWrapper(reader, encoding="utf-8"):
            if line.strip():
                yield json.loads(line)

In [ ]:
%%writefile src/ruencod/corpus.py
from pathlib import Path
from typing import Iterator

from ruencod.jsonl import iter_jsonl_zst


def domain_file(base: Path, split: str, domain: str) -> Path:
    return Path(base) / split / f"{domain}.jsonl.zst"


def iter_domain(base: Path, split: str, domain: str, max_bytes: int | None = None) -> Iterator[str]:
    path = domain_file(base, split, domain)
    if not path.exists():
        return
    used = 0
    for rec in iter_jsonl_zst(path):
        if max_bytes is not None and used >= max_bytes:
            return
        text = rec["text"]
        used += len(text.encode("utf-8"))
        yield text


def iter_mix(base: Path, split: str, shares: dict[str, float], max_bytes: int | None = None) -> Iterator[str]:
    for domain, share in shares.items():
        budget = None if max_bytes is None else int(max_bytes * share)
        yield from iter_domain(base, split, domain, budget)


def domain_shares(cfg: dict) -> dict[str, float]:
    return {d: v["share"] for d, v in cfg["corpus"]["domains"].items()}


def russian_domains(cfg: dict) -> list[str]:
    return [d for d, v in cfg["corpus"]["domains"].items() if v.get("russian", True)]

In [ ]:
%%writefile src/ruencod/pretok.py
import regex
from tokenizers import Regex, pre_tokenizers

_SUFFIX = r"(?:'[sS]|'[tT]|'[rR][eE]|'[vV][eE]|'[mM]|'[lL][lL]|'[dD])?"
# o200k-подобная регулярка: без inline-флагов (переносима в JS), цифры по одной
PATTERN = (
    r"[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+" + _SUFFIX
    + r"|[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*" + _SUFFIX
    + r"|\p{N}"
    + r"| ?[^\s\p{L}\p{N}]+[\r\n/]*"
    + r"|\s*[\r\n]+"
    + r"|\s+(?!\S)"
    + r"|\s+"
)
MARK = ""  # служебный разделитель кусков при обучении, вычищается из корпуса

_RE = regex.compile(PATTERN)


def pretokenize(text: str) -> list[str]:
    return _RE.findall(text)


def hf_pre_tokenizer():
    return pre_tokenizers.Sequence([
        pre_tokenizers.Split(Regex(PATTERN), "isolated"),
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
    ])


def hf_train_pre_tokenizer():
    return pre_tokenizers.Sequence([
        pre_tokenizers.Split(MARK, "removed"),
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
    ])

In [ ]:
%%writefile src/ruencod/data/clean.py
import html
import re
import unicodedata

from ruencod.pretok import MARK

_BLOCK_TAG = re.compile(r"</?(?:p|br|div|li|h[1-6]|blockquote|tr)\b[^<>]*>", re.I)
_TAG = re.compile(r"<[a-zA-Z/!][^<>]{0,500}>")
_MULTI_NL = re.compile(r"\n{3,}")


def _strip_html(text: str) -> str:
    text = _BLOCK_TAG.sub("\n\n", text)
    text = _TAG.sub("", text)
    text = html.unescape(text)
    return _MULTI_NL.sub("\n\n", text)


def normalize(text: str, html_source: bool = False) -> str:
    text = unicodedata.normalize("NFC", text)
    if html_source:
        text = _strip_html(text)
    return text.replace(MARK, "").strip()


def cyrillic_ratio(text: str) -> float:
    letters = cyr = 0
    for ch in text:
        if ch.isalpha():
            letters += 1
            if "Ѐ" <= ch <= "ӿ":
                cyr += 1
    return cyr / letters if letters else 0.0


def accept(text: str, *, min_chars: int, max_chars: int, russian: bool, min_cyrillic_ratio: float) -> bool:
    if not (min_chars <= len(text) <= max_chars):
        return False
    return not russian or cyrillic_ratio(text) >= min_cyrillic_ratio


def chunk_long(text: str, max_chars: int) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    chunks: list[str] = []
    cur: list[str] = []
    cur_len = 0
    for para in text.split("\n\n"):
        while len(para) > max_chars:
            if cur:
                chunks.append("\n\n".join(cur))
                cur, cur_len = [], 0
            chunks.append(para[:max_chars])
            para = para[max_chars:]
        add = len(para) + (2 if cur else 0)
        if cur and cur_len + add > max_chars:
            chunks.append("\n\n".join(cur))
            cur, cur_len = [para], len(para)
        else:
            cur.append(para)
            cur_len += add
    if cur:
        chunks.append("\n\n".join(cur))
    return [c for c in chunks if c.strip()]

In [ ]:
%%writefile src/ruencod/data/sources.py
import io
import json
import random
from typing import Iterator
from urllib.parse import urlparse

import pyarrow.compute as pc
import pyarrow.parquet as pq
import zstandard as zstd


def url_matches(url: str | None, domains: list[str]) -> bool:
    host = (urlparse(url).hostname or "") if url else ""
    return any(host == d or host.endswith("." + d) for d in domains)


def pikabu_docs(rec: dict) -> list[str]:
    docs = []
    post = "\n\n".join(x for x in (rec.get("title") or "", rec.get("text_markdown") or "") if x.strip())
    if post:
        docs.append(post)
    comments = [c for c in ((rec.get("comments") or {}).get("text_markdown") or []) if c and c.strip()]
    if comments:
        docs.append("\n".join(comments))
    return docs


def _iter_hf(src: dict, seed: int) -> Iterator[str]:
    from datasets import load_dataset

    ds = load_dataset(src["path"], src.get("name"), split=src.get("split", "train"), streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=src.get("shuffle_buffer", 10_000))
    flt = src.get("filter") or {}
    url_domains = src.get("url_domains")
    for row in ds:
        if any(row.get(k) != v for k, v in flt.items()):
            continue
        if url_domains and not url_matches(row.get("url"), url_domains):
            continue
        text = row.get(src["text"])
        if text:
            yield text


def _iter_pikabu(seed: int) -> Iterator[str]:
    from huggingface_hub import HfFileSystem

    fs = HfFileSystem()
    files = sorted(fs.glob("datasets/IlyaGusev/pikabu/*.jsonl.zst"))
    random.Random(seed).shuffle(files)
    for path in files:
        with fs.open(path, "rb") as f:
            reader = zstd.ZstdDecompressor().stream_reader(f)
            for line in io.TextIOWrapper(reader, encoding="utf-8"):
                if line.strip():
                    yield from pikabu_docs(json.loads(line))


def _iter_parquet(src: dict, seed: int, fs=None) -> Iterator[str]:
    """Читает из parquet только нужные колонки (row group целиком со всеми колонками не помещается в RAM)."""
    root = src["path"]
    if fs is None:  # по умолчанию — репозиторий датасета на HF Hub
        from huggingface_hub import HfFileSystem
        fs, root = HfFileSystem(), f"datasets/{src['path']}"
    files = sorted(fs.glob(f"{root}/{src.get('glob', '**/*.parquet')}"))
    rng = random.Random(seed)
    rng.shuffle(files)
    flt = src.get("filter") or {}
    columns = [src["text"], *flt]
    for path in files:
        with fs.open(path, "rb") as f:
            table = pq.read_table(f, columns=columns)
        for k, v in flt.items():
            table = table.filter(pc.equal(table[k], v))
        texts = table[src["text"]].to_pylist()
        rng.shuffle(texts)
        yield from (t for t in texts if t)


def iter_source(src: dict, seed: int, fs=None) -> Iterator[str]:
    kind = src.get("kind", "hf")
    if kind == "hf":
        yield from _iter_hf(src, seed)
    elif kind == "parquet":
        yield from _iter_parquet(src, seed, fs)
    elif kind == "pikabu":
        yield from _iter_pikabu(seed)
    else:
        raise ValueError(f"unknown source kind: {kind}")

In [ ]:
%%writefile src/ruencod/data/build.py
import argparse
import hashlib
import json
import os
import random

from tqdm import tqdm

from ruencod.config import DEFAULT_CONFIG, cfg_path, load_config
from ruencod.corpus import domain_file
from ruencod.data.clean import accept, chunk_long, normalize
from ruencod.data.sources import iter_source
from ruencod.jsonl import ZstJsonlWriter


def build(cfg: dict, domains: list[str] | None = None, iter_source_fn=iter_source) -> dict:
    corpus = cfg["corpus"]
    out = cfg_path(cfg, "data") / "corpus"
    seen: set[bytes] = set()
    stats: dict = {}
    for domain, dcfg in corpus["domains"].items():
        if domains and domain not in domains:
            continue
        train_target = int(corpus["total_bytes"] * dcfg["share"])
        held_target = int(corpus["heldout_bytes_per_domain"])
        srcs = dcfg["sources"]
        ds = {"train_bytes": 0, "heldout_bytes": 0, "docs": 0, "rejected": 0, "duplicates": 0, "shortfall": []}
        with ZstJsonlWriter(domain_file(out, "train", domain)) as wt, \
             ZstJsonlWriter(domain_file(out, "heldout", domain)) as wh:
            for i, src in enumerate(srcs):
                share = src.get("share", 1 / len(srcs))
                st, sh = int(train_target * share), int(held_target * share)
                p_held = sh / (st + sh) if st + sh else 0.0
                rng = random.Random(f"{cfg['seed']}:{domain}:{i}")
                bt = bh = 0
                bar = tqdm(total=st + sh, unit="B", unit_scale=True, desc=f"{domain}[{i}]")
                for raw in iter_source_fn(src, cfg["seed"]):
                    text = normalize(raw, html_source=src.get("html", False))
                    for chunk in chunk_long(text, corpus["max_chars"]):
                        if not accept(chunk, min_chars=corpus["min_chars"], max_chars=corpus["max_chars"],
                                      russian=dcfg.get("russian", True),
                                      min_cyrillic_ratio=corpus["min_cyrillic_ratio"]):
                            ds["rejected"] += 1
                            continue
                        h = hashlib.blake2b(chunk.encode("utf-8"), digest_size=16).digest()
                        if h in seen:
                            ds["duplicates"] += 1
                            continue
                        seen.add(h)
                        n = len(chunk.encode("utf-8"))
                        to_held = bh < sh and (bt >= st or rng.random() < p_held)
                        (wh if to_held else wt).write({"text": chunk, "domain": domain})
                        if to_held:
                            bh += n
                        else:
                            bt += n
                        ds["docs"] += 1
                        bar.update(n)
                        if bt >= st and bh >= sh:
                            break
                    if bt >= st and bh >= sh:
                        break  # не ждём следующий документ: у потоковых источников это могут быть минуты
                bar.close()
                if bt < st or bh < sh:
                    ds["shortfall"].append({"source": src.get("path", src.get("kind")),
                                            "train_missing": max(0, st - bt),
                                            "heldout_missing": max(0, sh - bh)})
                ds["train_bytes"] += bt
                ds["heldout_bytes"] += bh
        stats[domain] = ds
    stats_path = out / "stats.json"
    old = json.loads(stats_path.read_text(encoding="utf-8")) if stats_path.exists() else {}
    old.update(stats)
    stats_path.write_text(json.dumps(old, ensure_ascii=False, indent=2), encoding="utf-8")
    return stats


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=str(DEFAULT_CONFIG))
    ap.add_argument("--domains", nargs="*")
    args = ap.parse_args()
    stats = build(load_config(args.config), args.domains)
    print(json.dumps(stats, ensure_ascii=False, indent=2), flush=True)
    # потоковые загрузчики datasets оставляют фоновые потоки, из-за которых интерпретатор
    # может зависнуть или упасть при выходе; всё уже записано на диск — выходим сразу
    os._exit(0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/ruencod/eval/adapters.py
import os
from dataclasses import dataclass, field
from pathlib import Path

from tokenizers import Tokenizer

from ruencod.config import cfg_path

THREADS = os.cpu_count() or 4


@dataclass
class HFTok:
    name: str
    tok: Tokenizer

    @property
    def vocab_size(self) -> int:
        return self.tok.get_vocab_size()

    @property
    def bos_id(self):
        return self.tok.token_to_id("<|bos|>")

    def encode(self, text):
        return self.tok.encode(text, add_special_tokens=False).ids

    def encode_batch(self, texts):
        return [e.ids for e in self.tok.encode_batch(list(texts), add_special_tokens=False)]

    def decode(self, ids):
        return self.tok.decode(ids, skip_special_tokens=False)


@dataclass
class TiktokenTok:
    name: str
    encoding_name: str
    enc: object = field(init=False)

    def __post_init__(self):
        import tiktoken
        self.enc = tiktoken.get_encoding(self.encoding_name)

    @property
    def vocab_size(self):
        return self.enc.n_vocab

    @property
    def bos_id(self):
        return self.enc.eot_token

    def encode(self, text):
        return self.enc.encode_ordinary(text)

    def encode_batch(self, texts):
        return self.enc.encode_ordinary_batch(list(texts), num_threads=THREADS)

    def decode(self, ids):
        return self.enc.decode(ids)


@dataclass
class TransformersTok:
    name: str
    repo: str
    tk: object = field(init=False)

    def __post_init__(self):
        from transformers import AutoTokenizer
        self.tk = AutoTokenizer.from_pretrained(self.repo, trust_remote_code=False)

    @property
    def vocab_size(self):
        return len(self.tk)

    @property
    def bos_id(self):
        return self.tk.bos_token_id

    def encode(self, text):
        return self.tk.encode(text, add_special_tokens=False)

    def encode_batch(self, texts):
        return self.tk(list(texts), add_special_tokens=False)["input_ids"]

    def decode(self, ids):
        return self.tk.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)


@dataclass
class SentencePieceTok:
    name: str
    repo: str
    file: str
    sp: object = field(init=False)

    def __post_init__(self):
        import sentencepiece as spm
        from huggingface_hub import hf_hub_download
        self.sp = spm.SentencePieceProcessor(model_file=hf_hub_download(self.repo, self.file))

    @property
    def vocab_size(self):
        return self.sp.get_piece_size()

    @property
    def bos_id(self):
        return self.sp.bos_id()

    def encode(self, text):
        return self.sp.encode(text)

    def encode_batch(self, texts):
        return self.sp.encode(list(texts), num_threads=THREADS)

    def decode(self, ids):
        return self.sp.decode(ids)


def load_ours(path: Path) -> HFTok:
    path = Path(path)
    return HFTok(path.name, Tokenizer.from_file(str(path / "tokenizer.json")))


def list_ours(cfg: dict) -> list[Path]:
    root = cfg_path(cfg, "artifacts") / "tokenizers"
    return sorted(p for p in root.glob("*") if (p / "tokenizer.json").exists()) if root.exists() else []


def load_baseline(spec: dict):
    kind = spec["kind"]
    if kind == "tiktoken":
        return TiktokenTok(spec["name"], spec["id"])
    if kind == "hf":
        return TransformersTok(spec["name"], spec["id"])
    if kind == "sentencepiece":
        return SentencePieceTok(spec["name"], spec["id"], spec["file"])
    raise ValueError(kind)


def resolve(cfg: dict, name: str):
    ours = cfg_path(cfg, "artifacts") / "tokenizers" / name
    if (ours / "tokenizer.json").exists():
        return load_ours(ours)
    for spec in cfg["eval"]["baselines"]:
        if spec["name"] == name:
            return load_baseline(spec)
    raise KeyError(name)

In [ ]:
%%writefile src/ruencod/lm/model.py
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.checkpoint import checkpoint


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 384
    n_layers: int = 6
    n_heads: int = 6
    context: int = 512
    loss_chunk: int = 0  # >0: считать softmax кусками по столько токенов (экономия памяти на больших словарях)
    checkpoint_blocks: bool = False  # не хранить активации блоков, пересчитывать на backward (экономия памяти)


def rope_cache(seq_len: int, head_dim: int, base: float = 10000.0):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(seq_len).float(), inv)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    t = x.shape[-2]
    c, s = cos[:t], sin[:t]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack((x1 * c - x2 * s, x1 * s + x2 * c), dim=-1).flatten(-2)


class Block(nn.Module):
    def __init__(self, c: LMConfig):
        super().__init__()
        d = c.d_model
        self.h = c.n_heads
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        self.fc1 = nn.Linear(d, 4 * d, bias=False)
        self.fc2 = nn.Linear(4 * d, d, bias=False)

    def forward(self, x, cos, sin):
        b, t, c = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(c, dim=2)
        q, k, v = (z.view(b, t, self.h, c // self.h).transpose(1, 2) for z in (q, k, v))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(b, t, c)
        x = x + self.proj(y)
        return x + self.fc2(F.gelu(self.fc1(self.ln2(x))))


def _chunk_ce(h, y, w):
    return F.cross_entropy((h @ w.T).float(), y, reduction="sum")


class TinyLM(nn.Module):
    def __init__(self, c: LMConfig):
        super().__init__()
        self.c = c
        self.emb = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList(Block(c) for _ in range(c.n_layers))
        self.ln_f = nn.LayerNorm(c.d_model)
        cos, sin = rope_cache(c.context, c.d_model // c.n_heads)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        for p in self.parameters():
            if p.dim() == 2:
                nn.init.normal_(p, std=0.02)

    def forward(self, idx, targets=None, reduction="mean"):
        x = self.emb(idx)
        recompute = self.c.checkpoint_blocks and torch.is_grad_enabled()
        for blk in self.blocks:
            if recompute:
                x = checkpoint(blk, x, self.cos, self.sin, use_reentrant=False)
            else:
                x = blk(x, self.cos, self.sin)
        h = self.ln_f(x)
        if targets is None:
            return h @ self.emb.weight.T, None  # связанные эмбеддинги
        if not self.c.loss_chunk:
            logits = h @ self.emb.weight.T
            loss = F.cross_entropy(logits.view(-1, self.c.vocab_size).float(), targets.reshape(-1), reduction=reduction)
            return logits, loss
        # логиты по всему словарю для всего батча не держим: считаем кусками и пересчитываем на backward
        flat_h, flat_y = h.reshape(-1, h.shape[-1]), targets.reshape(-1)
        total = sum(
            checkpoint(_chunk_ce, flat_h[i:i + self.c.loss_chunk], flat_y[i:i + self.c.loss_chunk],
                       self.emb.weight, use_reentrant=False)
            for i in range(0, flat_y.numel(), self.c.loss_chunk)
        )
        return None, total if reduction == "sum" else total / flat_y.numel()

    def non_embedding_params(self) -> int:
        return sum(p.numel() for n, p in self.named_parameters() if not n.startswith("emb"))

In [ ]:
%%writefile src/ruencod/lm/train_lm.py
import contextlib
import math
import os
import time
from pathlib import Path

import numpy as np
import torch


def encode_corpus(tok, texts) -> np.ndarray:
    parts = []
    batch = []
    for t in texts:
        batch.append(t)
        if len(batch) == 256:
            parts += [[tok.bos_id] + ids for ids in tok.encode_batch(batch)]
            batch = []
    if batch:
        parts += [[tok.bos_id] + ids for ids in tok.encode_batch(batch)]
    return np.fromiter((i for p in parts for i in p), dtype=np.uint32)


def pick_device(pref: str = "auto") -> str:
    if pref == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    return pref


def amp_dtype(device: str, precision: str = "auto"):
    """bf16 на Ampere/Ada и новее; на старых картах (T4, P100, V100) — fp16 с масштабированием loss;
    на CPU или при precision=fp32 — без смешанной точности (None)."""
    if device != "cuda" or precision == "fp32":
        return None
    if precision == "auto":
        return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    return {"bf16": torch.bfloat16, "fp16": torch.float16}[precision]


def autocast(device: str, dtype=torch.bfloat16):
    if device == "cuda" and dtype is not None:
        return torch.autocast(device_type="cuda", dtype=dtype)
    return contextlib.nullcontext()


def save_checkpoint(path, model, opt, scaler, step: int, elapsed: float, loss: float, n_chunks: int) -> None:
    """Атомарно: при обрыве во время записи старое сохранение остаётся целым."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "step": step,
                "elapsed": elapsed, "loss": loss, "n_chunks": n_chunks}, tmp)
    os.replace(tmp, path)


def train_lm(model, tokens: np.ndarray, lm_cfg: dict, seed: int, log_every: int = 50,
             ckpt_path=None, resume: bool = False) -> dict:
    """ckpt_path: куда сохраняться раз в lm_cfg["ckpt_minutes"] минут, при Ctrl+C и в конце;
    resume: продолжить с сохранения (порядок батчей задан seed, так что итог тот же, что без перерыва)."""
    torch.manual_seed(seed)
    device = pick_device(lm_cfg.get("device", "auto"))
    if device == "cpu":
        torch.set_num_threads(lm_cfg["threads"])
    else:
        torch.cuda.reset_peak_memory_stats()  # пик — только этого кандидата
    model.to(device)
    ctx, bs = lm_cfg["context"], lm_cfg["batch_size"]
    n_chunks = (len(tokens) - 1) // ctx
    order = np.random.default_rng(seed).permutation(n_chunks)
    steps = math.ceil(n_chunks / bs)
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": lm_cfg["weight_decay"]},
                             {"params": no_decay, "weight_decay": 0.0}], lr=lm_cfg["lr"], betas=(0.9, 0.95),
                             fused=device == "cuda")
    dtype = amp_dtype(device, lm_cfg.get("precision", "auto"))
    scaler = torch.amp.GradScaler("cuda", enabled=dtype is torch.float16)  # для bf16 и fp32 — пустая обёртка
    warmup = max(1, int(steps * lm_cfg["warmup_frac"]))
    lr_max, lr_min = lm_cfg["lr"], lm_cfg["lr"] * 0.1
    start, before, loss_val = 0, 0.0, float("nan")
    if resume and ckpt_path and Path(ckpt_path).exists():
        st = torch.load(ckpt_path, map_location=device)
        if st["n_chunks"] != n_chunks:
            raise ValueError(f"{ckpt_path}: сохранение от другого корпуса ({st['n_chunks']} != {n_chunks} окон)")
        model.load_state_dict(st["model"])
        opt.load_state_dict(st["opt"])
        if st.get("scaler"):
            scaler.load_state_dict(st["scaler"])
        start, before, loss_val = st["step"], st["elapsed"], st["loss"]
        print(f"resume from step {start}/{steps}", flush=True)
    t0 = last_save = time.time()
    every = lm_cfg.get("ckpt_minutes", 10) * 60

    def save(done_steps: int) -> None:
        save_checkpoint(ckpt_path, model, opt, scaler, done_steps, before + time.time() - t0, loss_val, n_chunks)

    model.train()
    next_step = start
    try:
        for step in range(start, steps):
            idx = order[step * bs:(step + 1) * bs]
            win = np.stack([tokens[i * ctx:i * ctx + ctx + 1] for i in idx]).astype(np.int64)
            x, y = torch.from_numpy(win[:, :-1]).to(device), torch.from_numpy(win[:, 1:]).to(device)
            if step < warmup:
                lr = lr_max * (step + 1) / warmup
            else:
                prog = (step - warmup) / max(1, steps - warmup)
                lr = lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * prog))
            for g in opt.param_groups:
                g["lr"] = lr
            with autocast(device, dtype):
                _, loss = model(x, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            next_step = step + 1
            loss_val = loss.item()
            if step % log_every == 0:
                done = (step + 1 - start) * bs * ctx
                print(f"step {step}/{steps} loss {loss_val:.3f} tok/s {done / (time.time() - t0):.0f}", flush=True)
            if ckpt_path and time.time() - last_save >= every:
                save(next_step)
                last_save = time.time()
    except KeyboardInterrupt:
        if ckpt_path:
            save(next_step)  # незавершённый шаг начнётся заново
            print(f"saved {ckpt_path} at step {next_step}/{steps}", flush=True)
        raise
    if ckpt_path and start < steps:
        save(steps)  # обрыв во время оценки не потребует обучать заново
    secs = before + time.time() - t0
    trained = n_chunks * ctx
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else None
    return {"device": device, "steps": steps, "train_tokens": trained, "seconds": round(secs, 1),
            "tokens_per_sec": trained / secs if secs else 0.0, "final_loss": loss_val, "peak_gpu_gb": peak,
            "precision": {torch.bfloat16: "bf16", torch.float16: "fp16"}.get(dtype, "fp32")}

In [ ]:
%%writefile src/ruencod/lm/eval_bpb.py
import math

import torch

from ruencod.lm.train_lm import autocast


@torch.no_grad()
def nll_bits_and_bytes(model, tok, texts, context: int, dtype=torch.bfloat16) -> tuple[float, int]:
    model.eval()
    device = next(model.parameters(), torch.empty(0)).device
    nll, nbytes = 0.0, 0
    for t in texts:
        ids = [tok.bos_id] + tok.encode(t)
        nbytes += len(t.encode("utf-8"))
        for s in range(0, len(ids) - 1, context):
            chunk = ids[s:s + context + 1]
            x = torch.tensor(chunk[:-1], device=device).unsqueeze(0)
            y = torch.tensor(chunk[1:], device=device).unsqueeze(0)
            with autocast(device.type, dtype):
                _, loss = model(x, y, reduction="sum")
            nll += loss.item()
    return nll / math.log(2), nbytes

In [ ]:
%%writefile src/ruencod/lm/run.py
import argparse
import json
import os

# до первого обращения к CUDA: меньше фрагментации видеопамяти (на 8 ГБ это ~1 ГБ запаса)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import pandas as pd

from ruencod.config import DEFAULT_CONFIG, cfg_path, load_config
from ruencod.corpus import domain_shares, iter_domain, iter_mix, russian_domains
from ruencod.eval.adapters import resolve
from ruencod.lm.eval_bpb import nll_bits_and_bytes
from ruencod.lm.model import LMConfig, TinyLM
from ruencod.lm.train_lm import amp_dtype, encode_corpus, train_lm


def run_candidate(cfg: dict, name: str, train_bytes: int, ckpt_path=None, resume: bool = False) -> list[dict]:
    lm = cfg["lm"]
    tok = resolve(cfg, name)
    data = cfg_path(cfg, "data") / "corpus"
    tokens = encode_corpus(tok, iter_mix(data, "train", domain_shares(cfg), train_bytes))
    model = TinyLM(LMConfig(vocab_size=tok.vocab_size, d_model=lm["d_model"], n_layers=lm["n_layers"],
                            n_heads=lm["n_heads"], context=lm["context"], loss_chunk=lm.get("loss_chunk", 0),
                            checkpoint_blocks=lm.get("checkpoint_blocks", False)))
    stats = train_lm(model, tokens, lm, cfg["seed"], ckpt_path=ckpt_path, resume=resume)
    dtype = amp_dtype(stats["device"], lm.get("precision", "auto"))
    rows, ru_bits, ru_bytes = [], 0.0, 0
    for d in cfg["corpus"]["domains"]:
        texts = list(iter_domain(data, "heldout", d, lm["eval_bytes_per_domain"]))
        bits, nbytes = nll_bits_and_bytes(model, tok, texts, lm["context"], dtype)
        rows.append({"candidate": name, "domain": d, "bpb": bits / nbytes, **stats})
        if d in russian_domains(cfg):
            ru_bits, ru_bytes = ru_bits + bits, ru_bytes + nbytes
    rows.append({"candidate": name, "domain": "ru_all", "bpb": ru_bits / ru_bytes, **stats})
    return rows


def _smoke_line(name: str, rows: list[dict], full_bytes: int) -> None:
    r = rows[-1]
    est_h = r["train_tokens"] * (full_bytes / 1_000_000) / r["tokens_per_sec"] / 3600
    print(json.dumps({"candidate": name, "device": r["device"], "tokens_per_sec": round(r["tokens_per_sec"]),
                      "peak_gpu_gb": r["peak_gpu_gb"], "estimated_full_hours": round(est_h, 2)}), flush=True)


def run_all(cfg: dict, names: list[str], smoke: bool = False, resume: bool = False) -> pd.DataFrame:
    """resume: кандидаты, уже записанные в CSV, пропускаются; прерванный продолжается с сохранения."""
    full_bytes = cfg["lm"]["train_bytes"]
    out = cfg_path(cfg, "results") / ("lm_bpb_smoke.csv" if smoke else "lm_bpb.csv")
    ckpt_dir = cfg_path(cfg, "data") / "lm_ckpt"
    all_rows = pd.read_csv(out).to_dict("records") if resume and out.exists() else []
    finished = {r["candidate"] for r in all_rows}
    for name in names:
        if name in finished:
            print(f"{name}: уже в {out.name}, пропускаю", flush=True)
            continue
        ckpt = None if smoke else ckpt_dir / f"{name}.pt"
        rows = run_candidate(cfg, name, 1_000_000 if smoke else full_bytes, ckpt_path=ckpt, resume=resume)
        all_rows += rows
        if smoke:
            _smoke_line(name, rows, full_bytes)
        out.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(all_rows).to_csv(out, index=False)  # сохраняем после каждого кандидата
        if ckpt is not None:
            ckpt.unlink(missing_ok=True)  # результат записан — сохранение больше не нужно
    return pd.DataFrame(all_rows)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=str(DEFAULT_CONFIG))
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--resume", action="store_true", help="продолжить прерванный прогон")
    ap.add_argument("--candidates", nargs="*")
    args = ap.parse_args()
    cfg = load_config(args.config)
    df = run_all(cfg, args.candidates or cfg["lm"]["candidates"], smoke=args.smoke, resume=args.resume)
    print(df.query("domain == 'ru_all'")[["candidate", "bpb", "train_tokens", "seconds"]])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile configs/experiments.yaml
seed: 42
paths:
  data: data
  artifacts: artifacts
  results: results

corpus:
  total_bytes: 5000000000
  heldout_bytes_per_domain: 50000000
  min_chars: 200
  max_chars: 100000
  min_cyrillic_ratio: 0.6
  domains:
    web:
      share: 0.30
      russian: true
      sources:
        - {kind: hf, path: HuggingFaceFW/fineweb-2, name: rus_Cyrl, split: train, text: text}
    wiki:
      share: 0.15
      russian: true
      sources:
        - {kind: hf, path: wikimedia/wikipedia, name: 20231101.ru, split: train, text: text}
    social:
      share: 0.15
      russian: true
      sources:
        - {kind: pikabu}
    it:
      share: 0.10
      russian: true
      sources:
        - {kind: parquet, path: volosati/habr, glob: "data/*.parquet", text: text_markdown, filter: {language: ru}}
    books:
      share: 0.10
      russian: true
      sources:
        - {kind: hf, path: wikimedia/wikisource, name: 20231201.ru, split: train, text: text, shuffle_buffer: 1000}
    news:
      share: 0.10
      russian: true
      sources:
        - {kind: hf, path: IlyaGusev/gazeta, split: train, text: text, share: 0.5}
        - {kind: hf, path: zloelias/lenta-ru, split: train, text: text, share: 0.5}
    niche:
      share: 0.05
      russian: true
      sources:
        - {kind: parquet, path: irlspbru/RusLawOD, glob: "*.parquet", text: textIPS}
    en_code:
      share: 0.05
      russian: false
      sources:
        - {kind: hf, path: HuggingFaceFW/fineweb-edu, name: sample-10BT, split: train, text: text, share: 0.5}
        - {kind: hf, path: bigcode/the-stack-smol-xl, split: train, text: content, share: 0.5}

morph:
  tikhonov_urls:
    - https://raw.githubusercontent.com/AlexeySorokin/NeuralMorphemeSegmentation/master/data/train_Tikhonov_reformat.txt
    - https://raw.githubusercontent.com/AlexeySorokin/NeuralMorphemeSegmentation/master/data/test_Tikhonov_reformat.txt
  test_frac: 0.1
  min_word_freq: 5
  workers: 8

train:
  vocab_sizes: [16000, 32000, 48000, 64000, 96000]
  algos: [bpe, unigram, morph-bpe]
  bpe_max_bytes: null          # null = весь train; уменьшить при нехватке RAM
  unigram_max_bytes: 1500000000
  bpe_min_frequency: 2

eval:
  max_bytes_per_domain: 50000000
  claude:
    models: []                 # без ключа API Claude не считаем; вернуть [claude-opus-5] и docs_per_domain: 200
    docs_per_domain: 0
  baselines:
    - {name: o200k, label: "GPT-4o/GPT-5 (o200k)", kind: tiktoken, id: o200k_base}
    - {name: cl100k, label: "GPT-4 (cl100k)", kind: tiktoken, id: cl100k_base}
    - {name: llama-3, label: "Llama 3", kind: hf, id: NousResearch/Meta-Llama-3-8B}
    - {name: qwen-3, label: "Qwen 3", kind: hf, id: Qwen/Qwen3-8B}
    - {name: mistral-nemo, label: "Mistral Nemo", kind: hf, id: unsloth/Mistral-Nemo-Base-2407}
    - {name: gemma-3, label: "Gemma 3", kind: hf, id: unsloth/gemma-3-1b-it}
    - {name: rugpt-3, label: "ruGPT-3", kind: hf, id: ai-forever/rugpt3large_based_on_gpt2}
    - {name: vikhr, label: "Vikhr 7B", kind: hf, id: Vikhrmodels/Vikhr-7B-instruct_0.4}
    - {name: gigachat, label: "GigaChat (open)", kind: hf, id: ai-sage/GigaChat-20B-A3B-instruct}
    - {name: yandexgpt-5-lite, label: "YandexGPT 5 Lite", kind: sentencepiece, id: yandex/YandexGPT-5-Lite-8B-pretrain, file: tokenizer.model}

prices:
  verified_on: "2026-10-12"
  gpt:
    label: "GPT-4o"
    tokenizer: o200k
    usd_per_mtok: 2.50
    source: https://developers.openai.com/api/docs/models/gpt-4o
  claude:
    label: "Claude Opus 5"
    tokenizer: claude-opus-5
    usd_per_mtok: 5.00
    source: https://docs.claude.com/en/docs/about-claude/pricing

lm:
  context: 512
  d_model: 384
  n_layers: 6
  n_heads: 6
  batch_size: 16
  lr: 0.001
  warmup_frac: 0.02
  weight_decay: 0.1
  train_bytes: 30000000        # пересчитывается по smoke-run
  eval_bytes_per_domain: 2000000
  threads: 20
  candidates: [bpe-64k, unigram-64k, morph-bpe-64k, o200k]

final:
  tokenizer: bpe-64k           # V=64k выбран пользователем; финалист уточняется по мини-LM

examples:
  - "достопримечательностями"
  - "переподготовка"
  - "Новая нейросеть пишет код"
  - "1234567"
  - "Я написал скрипт на Python, который парсит JSON и шлёт в Telegram."
  - "Нейросеть не читает буквы — она видит токены."

In [ ]:
%%writefile configs/gpu.yaml
# Мини-модели на видеокарте NVIDIA 8 ГБ (RTX 4060 Ti): uv sync --extra cu126
base: experiments.yaml
lm:
  device: auto              # cuda, если есть, иначе cpu
  d_model: 768              # ~85M параметров без эмбеддингов (масштаб GPT-2 small)
  n_layers: 12
  n_heads: 12
  context: 512
  batch_size: 8             # 4096 токенов на шаг; o200k: ~3.8 ГБ весов и состояний Adam + активации ≈ 5.2 ГБ
  loss_chunk: 512           # softmax по словарю считается кусками по 512 токенов
  checkpoint_blocks: true   # активации блоков пересчитываются на backward: o200k 6.8 → 5.2 ГБ, скорость та же;
                            # без этого на 8 ГБ (минус ~1.5 ГБ рабочего стола Windows) o200k уходит в общую память и в 8× медленнее
  lr: 0.0006
  warmup_frac: 0.02
  weight_decay: 0.1
  train_bytes: 300000000    # 300 МБ текста на кандидата; уточнить по --smoke
  eval_bytes_per_domain: 2000000
  ckpt_minutes: 10          # сохранение в data/lm_ckpt/ для --resume (o200k: ~2.9 ГБ на диске)
  candidates: [bpe-64k, unigram-64k, morph-bpe-64k, o200k]

Облачный конфиг: модель, данные, скорость обучения, размер батча — те же, что дома (`gpu.yaml`).
Меняются только пути и то, что влияет лишь на память и скорость, но не на результат:
пересчёт активаций нужен только картам меньше 12 ГБ, softmax кусками — меньше 30 ГБ.

In [ ]:
cloud = f'''# собрано ноутбуком: модель и данные как дома; меняются только пути и расход памяти
base: gpu.yaml
paths:
  data: "{DATA_DIR.as_posix()}"
  results: "{RESULTS_DIR.as_posix()}"
lm:
  precision: {precision}
  checkpoint_blocks: {"true" if vram_gb < 12 else "false"}
  loss_chunk: {512 if vram_gb < 30 else 0}
  candidates: [{CANDIDATE}]
'''
(WORK / "configs/cloud.yaml").write_text(cloud, encoding="utf-8")
print(cloud)

## 5. Корпус

Те же источники, seed и бюджеты, что дома: ~5 ГБ обучающего текста из 8 доменов + по 50 МБ отложенного.
Модели нужны только первые 300 МБ смеси, но разбиение train/heldout зависит от полного объёма,
поэтому собирается всё. Если корпус уже собран (например, на Google Диске), шаг пропускается.

In [ ]:
corpus = DATA_DIR / "corpus"
domains = ["web", "wiki", "social", "it", "books", "news", "niche", "en_code"]
ready = (corpus / "stats.json").exists() and all(
    (corpus / split / f"{d}.jsonl.zst").exists() for split in ("train", "heldout") for d in domains)
if ready:
    print("Корпус уже собран:", corpus)
else:
    run("ruencod.data.build", "--config", "configs/cloud.yaml")
    missing = [f"{split}/{d}" for split in ("train", "heldout") for d in domains
               if not (corpus / split / f"{d}.jsonl.zst").exists()]
    if missing:
        raise RuntimeError(f"Сборка не создала: {missing} — причина в выводе выше")

## 6. Проверка данных

Отпечаток (SHA-256) ровно тех текстов, которые увидит модель: 300 МБ обучающей смеси и по 2 МБ отложенного
текста на домен. Ожидаемые значения посчитаны дома на данных, на которых учились остальные кандидаты.
Несовпадение почти наверняка значит, что какой-то датасет на Hugging Face обновился — тогда сравнение
с домашними кандидатами становится менее строгим (`ALLOW_DATA_MISMATCH = True`, чтобы всё равно продолжить).

In [ ]:
import hashlib

from ruencod.config import cfg_path, load_config
from ruencod.corpus import domain_shares, iter_domain, iter_mix

EXPECTED = {
    "train": {
        "sha256": "04478b28d4d5008a",
        "docs": 42086,
        "bytes": 300238239
    },
    "train/web": {
        "sha256": "ea4cc81b22412f6d",
        "docs": 9726,
        "bytes": 90056816
    },
    "train/wiki": {
        "sha256": "1e0ac618d2ae77d1",
        "docs": 9140,
        "bytes": 45001666
    },
    "train/social": {
        "sha256": "50c42fdf9ac0eb0c",
        "docs": 12067,
        "bytes": 45004859
    },
    "train/it": {
        "sha256": "f7b0090748049a4d",
        "docs": 2422,
        "bytes": 30001491
    },
    "train/books": {
        "sha256": "a92aa7d2a2afa2ac",
        "docs": 993,
        "bytes": 30101602
    },
    "train/news": {
        "sha256": "31074b2129cb4ba9",
        "docs": 3797,
        "bytes": 30002854
    },
    "train/niche": {
        "sha256": "9926974e748461ce",
        "docs": 915,
        "bytes": 15067211
    },
    "train/en_code": {
        "sha256": "d1d54355b2c300d0",
        "docs": 3026,
        "bytes": 15001740
    },
    "heldout/web": {
        "sha256": "16f093a10e5ea4a3",
        "docs": 215,
        "bytes": 2001002
    },
    "heldout/wiki": {
        "sha256": "61e4f5b065ec65e5",
        "docs": 466,
        "bytes": 2001493
    },
    "heldout/social": {
        "sha256": "f4808edc4692e2d0",
        "docs": 600,
        "bytes": 2004030
    },
    "heldout/it": {
        "sha256": "405d8fe630d17cc1",
        "docs": 163,
        "bytes": 2003076
    },
    "heldout/books": {
        "sha256": "89593ed83797e969",
        "docs": 78,
        "bytes": 2125306
    },
    "heldout/news": {
        "sha256": "b4dbe6c25ee70ab1",
        "docs": 248,
        "bytes": 2006454
    },
    "heldout/niche": {
        "sha256": "93b32de3e97b983e",
        "docs": 140,
        "bytes": 2001485
    },
    "heldout/en_code": {
        "sha256": "b525ad0a05912746",
        "docs": 418,
        "bytes": 2007709
    }
}


def digest(texts):
    h, n, b = hashlib.sha256(), 0, 0
    for t in texts:
        e = t.encode("utf-8")
        h.update(len(e).to_bytes(8, "little"))
        h.update(e)
        n, b = n + 1, b + len(e)
    return {"sha256": h.hexdigest()[:16], "docs": n, "bytes": b}


cfg = load_config(WORK / "configs/cloud.yaml")
data, lm = cfg_path(cfg, "data") / "corpus", cfg["lm"]
shares = domain_shares(cfg)
got = {"train": digest(iter_mix(data, "train", shares, lm["train_bytes"]))}
for d, share in shares.items():
    got[f"train/{d}"] = digest(iter_domain(data, "train", d, int(lm["train_bytes"] * share)))
for d in shares:
    got[f"heldout/{d}"] = digest(iter_domain(data, "heldout", d, lm["eval_bytes_per_domain"]))
empty = [k for k, v in got.items() if v["docs"] == 0]
if empty:
    raise RuntimeError(f"В {data} нет текстов для {empty}: корпус не собран — смотрите вывод шага 5.")
bad = [k for k in EXPECTED if got.get(k) != EXPECTED[k]]
for k in bad:
    print(f"{k}: ожидалось {EXPECTED[k]}, получено {got.get(k)}")
if not bad:
    print("Данные совпадают с домашними: 300 МБ обучающей смеси и 8 × 2 МБ отложенного текста.")
elif not ALLOW_DATA_MISMATCH:
    raise RuntimeError("Данные отличаются от домашних (см. выше). Чтобы всё равно обучать — ALLOW_DATA_MISMATCH = True.")

## 7. Пробный прогон

1 МБ текста: скорость (`tokens_per_sec`), пик видеопамяти и прогноз полного обучения (`estimated_full_hours`).
Большая часть времени этой ячейки — оценка на 16 МБ отложенного текста, а не обучение.

In [ ]:
if RUN_SMOKE:
    run("ruencod.lm.run", "--config", "configs/cloud.yaml", "--smoke", "--candidates", CANDIDATE)

## 8. Обучение

Строка в лог каждые 50 шагов: `step`, `loss` (на токен, сильно шумит — смотрите на тренд) и `tok/s`.
Раз в 10 минут — сохранение в `data/lm_ckpt/`; кнопка «Стоп» тоже сохраняет. Если среда отключится,
перезапуск ячейки продолжит с сохранения (`--resume`), результат тот же, что без перерыва.

In [ ]:
run("ruencod.lm.run", "--config", "configs/cloud.yaml", "--candidates", CANDIDATE, "--resume")

## 9. Результат

In [ ]:
import pandas as pd

res = pd.read_csv(RESULTS_DIR / "lm_bpb.csv")
res = res[res.candidate == CANDIDATE]
out = RESULTS_DIR / f"lm_bpb_{CANDIDATE}.csv"
res.to_csv(out, index=False)
display(res[["candidate", "domain", "bpb"]])
r = res[res.domain == "ru_all"].iloc[0]
print(f"{CANDIDATE}: bpb на русском {r.bpb:.4f} | {r.steps} шагов | {r.seconds / 3600:.2f} ч | {r.precision} | "
      f"пик {r.peak_gpu_gb:.1f} ГБ")
print("Дома: unigram-64k 0.9276, morph-bpe-64k 0.9342, bpe-64k 0.9735 (меньше — лучше)")
print("Файл:", out)
if IN_COLAB:
    from google.colab import files
    files.download(str(out))